# ME5.2: Bit Flip on $|\phi+\rangle$ (Time-dependent)

## Objectives
- Model time-dependent bit-flip noise $p(t)$ on a Bell state.
- Track how the correlation $\langle ZZ\rangle(t)$ decays with $t$.

## Setup
```python
import numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Kraus
```


## Theory Snapshot
- Independent flips with probability $p(t)$ on each qubit give $\langle ZZ\rangle(t)=(1−2p(t))^2$.
- If $p(t)=(1−e^{−2t/T_X})/2$ (chosen relaxation model), then $\langle ZZ\rangle(t)=e^{−4t/T_X}$.

## Experiment


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Kraus

# --- Helpers ---
def bit_flip_kraus(p):
    I = np.array([[1,0],[0,1]], complex)
    X = np.array([[0,1],[1,0]], complex)
    return Kraus([np.sqrt(1-p)*I, np.sqrt(p)*X])

def bell_phi_plus():
    qc = QuantumCircuit(2,2)
    qc.h(0); qc.cx(0,1)   # |phi+>
    return qc

def zz_corr(counts, shots):
    return (counts.get('00',0)+counts.get('11',0)-counts.get('01',0)-counts.get('10',0))/shots

# --- Parameters ---
T   = 30.0        # decay constant (microseconds)
tmax, dt = 3*T, 1.0
times = np.arange(0.0, tmax+1e-12, dt)
p_t   = (1 - np.exp(-2*times/T))/2.0  # bit-flip prob vs time

# --- Build circuits (bit-flip applied once with time-dependent p) ---
sim = AerSimulator(method="density_matrix")
shots = 10000
circs = []
for p in p_t:
    BF = bit_flip_kraus(float(p))
    qc = bell_phi_plus()
    qc.append(BF, [0]); qc.append(BF, [1])   # independent flips per qubit at time t
    qc.measure([0,1],[0,1])
    circs.append(qc)

# --- Run & collect correlations ---
tqcs = transpile(circs, sim)
res  = sim.run(tqcs, shots=shots).result()
zz_sim = [zz_corr(res.get_counts(i), shots) for i in range(len(times))]

# --- Theory: <ZZ>(t) = e^{-4 t / T} ---
zz_theory = np.exp(-4*times/T)

# --- Plot ---
plt.figure(figsize=(7,4))
plt.plot(times, zz_sim, marker='o', linestyle='', label='Simulated $\\langle Z \otimes Z\\rangle$')
plt.plot(times, zz_theory, linestyle='--', label='Theory $e^{-4t/T}$')
plt.xlabel(f'Time (microseconds)'); plt.ylabel('Correlation $\\langle Z \otimes Z\\rangle$')
plt.title('Bit-Flip Noise on $|\\phi⁺\\rangle$: Correlation Decay vs Time')
plt.ylim(-0.05, 1.05); plt.legend(); plt.tight_layout(); plt.show()


## Results & Discussion

### Results
- $\langle ZZ\rangle(t)$ decays smoothly and matches $e^{−4t/T}$.
- Small deviations arise from finite-shot sampling.

### Discussion
- Time-dependent bit-flip noise progressively destroys Bell-state parity.
- Agreement with $e^{−4t/T}$ confirms the chosen $p(t)$ model.